<a href="https://colab.research.google.com/github/Bhavanavaddadi/prediction-model--churn/blob/main/mode_for_churn_prrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We'll install and import everything we need for data handling, machine learning, and building our app

In [ ]:

!pip install -q pandas scikit-learn xgboost joblib gradio


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import joblib
import io
from google.colab import files
import gradio as gr

print("✅ All libraries are ready!")

✅ All libraries are ready!


Here, we'll use Colab's built-in tool to upload the CSV file from your computer.

In [ ]:

print("Please upload the 'WA_Fn-UseC_-Telco-Customer-Churn.csv' file.")
uploaded = files.upload()

Please upload the 'WA_Fn-UseC_-Telco-Customer-Churn.csv' file.


Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


We'll load the data and immediately fix a tricky column like TotalCharges, which sometimes has blank spaces instead of numbers. We'll convert it to a number and remove any rows that couldn't be converted.

In [ ]:
try:
    df = pd.read_csv(io.BytesIO(uploaded['WA_Fn-UseC_-Telco-Customer-Churn.csv']))
    print("\nFile uploaded and loaded successfully!")

    # Convert TotalCharges to a number. If it can't, it becomes 'NaN' (Not a Number).
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # drop any rows that have missing values (including the new NaNs)
    df.dropna(inplace=True)

    # We don't need the customerID for prediction, so let's remove it
    df = df.drop('customerID', axis=1)

    print("\nData cleaning complete. Here's a look at the data types now:")
    df.info() # You'll see TotalCharges is now a float64, which is correct!

except KeyError:
    print("\n🔴 ERROR: You must upload a file named 'WA_Fn-UseC_-Telco-Customer-Churn.csv'. Please restart and try again.")
except Exception as e:
    print(f"\n🔴 ERROR: An unexpected error occurred: {e}")


File uploaded and loaded successfully!

Data cleaning complete. Here's a look at the data types now:
<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   objec

Here, we'll convert the 'Churn' column to 0s and 1s. Then, we'll separate our data into features (X) and the target we want to predict (y), and finally split them into training and testing sets.

In [ ]:
# Convert the 'Churn' column from 'Yes'/'No' to 1/0
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])

# Separate our features (the inputs) from our target (the output)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify which columns are text and which are numbers
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

print(f"\nCategorical columns we'll encode: {list(categorical_features)}")
print(f"Numerical columns we'll scale: {list(numerical_features)}")

# Split the data so we can train on one part and test on another
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("\n✅ Data successfully split into training and testing sets.")


Categorical columns we'll encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns we'll scale: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

✅ Data successfully split into training and testing sets.


Building the training pipeline and training our model.

In [ ]:
# Create a preprocessor to handle text and number columns differently
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Now, create the full pipeline: Preprocess data, then classify
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss'))])

# Time to train! This is where the model learns from the data.
print("\nTraining the model... this might take a moment.")
model_pipeline.fit(X_train, y_train)
print("✅ Model training complete!")


Training the model... this might take a moment.
✅ Model training complete!


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [07:13:57] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saving our trained model.

In [ ]:
# Save the entire pipeline to a file
joblib.dump(model_pipeline, 'churn_model_pipeline.pkl')
print("\n✅ Model saved as 'churn_model_pipeline.pkl'. We're ready to build the app!")


✅ Model saved as 'churn_model_pipeline.pkl'. We're ready to build the app!


This function is the core logic of our dashboard. It will take all the inputs from the user interface, put them into the right format (a pandas DataFrame), and use our saved model to make a prediction.

In [ ]:
# This function connects our UI to our model
def predict_churn(gender, senior_citizen, partner, dependents, tenure,
                  phone_service, multiple_lines, internet_service,
                  online_security, online_backup, device_protection,
                  tech_support, streaming_tv, streaming_movies,
                  contract, paperless_billing, payment_method,
                  monthly_charges, total_charges):

    # Create a dictionary from the inputs
    data = {
        'gender': gender, 'SeniorCitizen': senior_citizen, 'Partner': partner,
        'Dependents': dependents, 'tenure': tenure, 'PhoneService': phone_service,
        'MultipleLines': multiple_lines, 'InternetService': internet_service,
        'OnlineSecurity': online_security, 'OnlineBackup': online_backup,
        'DeviceProtection': device_protection, 'TechSupport': tech_support,
        'StreamingTV': streaming_tv, 'StreamingMovies': streaming_movies,
        'Contract': contract, 'PaperlessBilling': paperless_billing,
        'PaymentMethod': payment_method, 'MonthlyCharges': monthly_charges,
        'TotalCharges': total_charges
    }

    # Convert the dictionary into a DataFrame, which our model expects
    input_df = pd.DataFrame(data, index=[0])

    # Use the loaded model to make a prediction
    prediction = model_pipeline.predict(input_df)
    prediction_proba = model_pipeline.predict_proba(input_df)

    # Get the specific probabilities
    prob_no_churn = prediction_proba[0][0]
    prob_churn = prediction_proba[0][1]

    # Create a nice, human-readable verdict
    verdict = "🟢 UNLIKELY TO CHURN" if prediction[0] == 0 else "🔴 LIKELY TO CHURN"

    # Format the probabilities for the output label
    probabilities = {'Probability of Not Churning': f"{prob_no_churn:.2f}",
                     'Probability of Churning': f"{prob_churn:.2f}"}

    return verdict, probabilities

print("✅ Prediction function is defined.")

✅ Prediction function is defined.


Designing the user interface and launching our app

In [ ]:
# --- This is where we design the Gradio UI ---
if 'model_pipeline' in locals():
    print("Building the Gradio interface...")
    # --- Inputs (with default values to prevent errors!) ---
    inputs = [
        gr.Dropdown(['Male', 'Female'], label="Gender", value='Male'),
        gr.Dropdown([('No', 0), ('Yes', 1)], label="Senior Citizen", value=0),
        gr.Dropdown(['Yes', 'No'], label="Partner", value='Yes'),
        gr.Dropdown(['Yes', 'No'], label="Dependents", value='No'),
        gr.Slider(1, 72, value=24, label="Tenure (months)"),
        gr.Dropdown(['Yes', 'No'], label="Phone Service", value='Yes'),
        gr.Dropdown(['Yes', 'No', 'No phone service'], label="Multiple Lines", value='No'),
        gr.Dropdown(['DSL', 'Fiber optic', 'No'], label="Internet Service", value='DSL'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Online Security", value='No'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Online Backup", value='No'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Device Protection", value='No'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Tech Support", value='No'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Streaming TV", value='No'),
        gr.Dropdown(['Yes', 'No', 'No internet service'], label="Streaming Movies", value='No'),
        gr.Dropdown(['Month-to-month', 'One year', 'Two year'], label="Contract", value='Month-to-month'),
        gr.Dropdown(['Yes', 'No'], label="Paperless Billing", value='Yes'),
        gr.Dropdown(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], label="Payment Method", value='Electronic check'),
        gr.Slider(18.0, 120.0, value=70.0, label="Monthly Charges ($)"),
        gr.Slider(18.0, 8700.0, value=1400.0, label="Total Charges ($)")
    ]

    # --- Outputs ---
    outputs = [
        gr.Textbox(label="Prediction Verdict"),
        gr.Label(label="Prediction Probabilities")
    ]

    # --- Putting it all together ---
    iface = gr.Interface(
        fn=predict_churn,
        inputs=inputs,
        outputs=outputs,
        title="📈 Telco Customer Churn Prediction",
        description="Enter a customer's details to predict their likelihood of churning. This dashboard uses a model trained with XGBoost.",
        allow_flagging="never"
    )

    # --- Launch the app! ---
    print("\nLaunching the app... Click the public URL link below.")
    iface.launch(share=True, debug=True)
else:
    print("🔴 ERROR: 'model_pipeline' is not defined. Please run the previous cells successfully first.")

Building the Gradio interface...


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(



Launching the app... Click the public URL link below.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f0b7d4552bb61e3312.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
